## Setup and Dependencies

In [7]:
!pip install openai datasets networkx numpy python-dotenv

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
import networkx as nx
import datasets
import json
import re
import os

Constants

In [47]:
load_dotenv()

client = OpenAI()
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: API key not found. Please check your .env file.")
else:
    print("API key loaded successfully.")

MODELS = {
    "4o-mini":  {"model": "gpt-4o-mini", "reasoning_lvl": None},
    "4o-base":  {"model": "gpt-4o",      "reasoning_lvl": None},
    "o3-mini":  {"model": "o3-mini",     "reasoning_lvl": "medium"} 
}

JUDGE_MODEL = "gpt-4o"

EXTRACTION_MODEL  = "gpt-4o-mini"   # cheap — runs once per question per paragraph
EMBEDDING_MODEL   = "text-embedding-3-small"
SIMILARITY_THRESHOLD = 0.2           # nodes below this vs the question get pruned

API key loaded successfully.


Prompts

In [48]:
SYS_PROMPT = """You are a strict, deterministic QA assistant. 
Answer the user's question using ONLY the provided context.
You must output ONLY the exact entity, name, date, or 'yes/no' that answers the question.
CRITICAL: Do NOT write full sentences. Do NOT include punctuation unless part of the name. Do NOT add conversational filler like 'The answer is' or 'Yes, the...'."""

JUDGE_PROMPT = """You are a strict but fair evaluator for factual question answering.

Question: {question}
Gold answer: {gold}
Predicted answer: {predicted}

Does the predicted answer correctly answer the question, matching the gold answer in meaning?
Minor phrasing differences are fine. Extra explanation is fine as long as the correct answer is present.

Respond ONLY with valid JSON:
{{"score": 0 or 1, "reason": "one sentence"}}"""

EXTRACTION_PROMPT = """Extract factual (subject, relation, object) triples from the text below.
Return a JSON object with a single key "triples" containing an array of objects.
Each object must have keys: "subject", "relation", "object".
Keep entities concise (no long phrases). Extract as many meaningful triples as you can.

Text: {text}"""

Dataset Initialization

In [49]:
def load_hotpotqa(n=30):
    data = datasets.load_dataset("hotpot_qa", "distractor", split=f"validation[0:{n}]")
    rows = []
    
    for row in data:
        paragraphs = []
        raw_context_string = "" 
        
        for title, sentences in zip(row["context"]["title"], row["context"]["sentences"]):
            para_text = " ".join(sentences)
            
            #  structured data for the Knowledge Graph pipeline
            paragraphs.append({
                "title"    : title,
                "text"     : para_text,
                "sentences": sentences,
            })
            
            # raw string for raw LLM
            raw_context_string += f"Title: {title}\nText: {para_text}\n\n"

        rows.append({
            "Q": row["question"],
            "A": row["answer"],
            "paragraphs": paragraphs,             # KG uses this
            "raw_context": raw_context_string     # Raw LLM uses this
        })
        
    return rows

my_ds = load_hotpotqa(n=25)
my_ds[0]

{'Q': 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'A': 'yes',
 'paragraphs': [{'title': 'Ed Wood (film)',
   'text': "Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.",
   'sentences': ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
    " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.",
    ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are amon

## RAW LLM (Stem Agent)

Answer Generation

In [50]:
def llm_call(question, context, model_key):
    cfg = MODELS[model_key]
    model = cfg["model"]
    reasoning_lvl = cfg["reasoning_lvl"]

    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    if reasoning_lvl is not None:
        response = client.chat.completions.create(
            model=model,
            max_completion_tokens=600,
            reasoning_effort=reasoning_lvl,
            messages=[
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user",   "content": user_prompt},
            ]
        )
        return response.choices[0].message.content.strip()

    else:
        # Standard — no reasoning
        response = client.chat.completions.create(
            model=model,
            max_completion_tokens=300,
            temperature=0.0,
            messages=[
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user",   "content": user_prompt},
            ]
        )
        return response.choices[0].message.content.strip()

Running the API call on all three models

In [51]:
def run_all(dataset):
    all_results = {}

    for model_key in MODELS:
        cfg = MODELS[model_key]
        print(f"\n{'='*60}")
        print(f"  {model_key.upper()} → {cfg['model']}  (reasoning: {cfg['reasoning_lvl'] or 'off'})")
        print(f"{'='*60}")

        preds = []
        for i, item in enumerate(dataset):
            try:
                pred = llm_call(item["Q"], item["raw_context"], model_key)
            except Exception as e:
                print(f"  [ERROR] {e}")
                pred = ""

            preds.append({
                "question"  : item["Q"],
                "gold"      : item["A"],
                "predicted" : pred,
            })

            print(f"[{i+1:02d}] Q: {item['Q'][:65]}")
            print(f"      Gold: {item['A']}")
            print(f"      Pred: {pred[:80]}\n")

        all_results[model_key] = {
            "model"         : cfg["model"],
            "reasoning_lvl" : cfg["reasoning_lvl"] or "off",
            "predictions"   : preds,
        }

    return all_results

all_results = run_all(my_ds)

with open("predictions.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Saved → predictions.json")


  4O-MINI → gpt-4o-mini  (reasoning: off)
[01] Q: Were Scott Derrickson and Ed Wood of the same nationality?
      Gold: yes
      Pred: yes

[02] Q: What government position was held by the woman who portrayed Corl
      Gold: Chief of Protocol
      Pred: ambassador

[03] Q: What science fantasy young adult series, told in first person, ha
      Gold: Animorphs
      Pred: Animorphs

[04] Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same
      Gold: no
      Pred: no

[05] Q: The director of the romantic comedy "Big Stone Gap" is based in w
      Gold: Greenwich Village, New York City
      Pred: Greenwich Village

[06] Q: 2014 S/S is the debut album of a South Korean boy group that was 
      Gold: YG Entertainment
      Pred: YG Entertainment

[07] Q: Who was known by his stage name Aladin and helped organizations i
      Gold: Eenasul Fateh
      Pred: Eenasul Fateh

[08] Q: The arena where the Lewiston Maineiacs played their home games ca
      Gold: 3,677 sea

Evaluation

In [ ]:
def normalize(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[^\w\s]', '', s)
    return re.sub(r'\s+', ' ', s).strip()

def exact_match(pred: str, gold: str) -> bool:
    return normalize(pred) == normalize(gold)

def llm_judge(question: str, gold: str, predicted: str) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question,
        gold=gold,
        predicted=predicted
    )
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        max_completion_tokens=120,
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"score": 0, "reason": "parse error"}

# %%
def evaluate(predictions_path: str) -> dict:
    with open(predictions_path) as f:
        all_results = json.load(f)

    scored_results = {}

    for model_key, data in all_results.items():
        print(f"\n{'='*60}")
        print(f"  Evaluating: {model_key.upper()} → {data['model']}")
        print(f"{'='*60}")

        em_scores, llm_scores, scored_preds = [], [], []

        for i, item in enumerate(data["predictions"]):
            q, gold, pred = item["question"], item["gold"], item["predicted"]

            em    = exact_match(pred, gold)
            judge = llm_judge(q, gold, pred)

            em_scores.append(int(em))
            llm_scores.append(judge["score"])

            scored_preds.append({
                **item,
                "exact_match" : int(em),
                "llm_score"   : judge["score"],
                "reason"      : judge["reason"],
            })

            print(f"[{i+1:02d}] EM: {int(em)} | Judge: {judge['score']} — {judge['reason']}")

        em_acc  = round(sum(em_scores)  / len(em_scores),  4)
        llm_acc = round(sum(llm_scores) / len(llm_scores), 4)

        print(f"\n  Exact Match : {em_acc:.1%}")
        print(f"  LLM Judge   : {llm_acc:.1%}")

        scored_results[model_key] = {
            "model"               : data["model"],
            "reasoning_lvl"       : data["reasoning_lvl"],
            "exact_match_accuracy": em_acc,
            "llm_judge_accuracy"  : llm_acc,
            "predictions"         : scored_preds,
        }

    return scored_results

scored = evaluate("predictions.json")

print(f"\n{'='*60}")
print(f"  BASELINE SUMMARY")
print(f"{'='*60}")
print(f"{'Tier':<12} {'Model':<16} {'Reasoning':<12} {'EM':>6} {'Judge':>7}")
print(f"{'-'*55}")
for key, res in scored.items():
    print(
        f"{key:<12} {res['model']:<16} {res['reasoning_lvl']:<12}"
        f"{res['exact_match_accuracy']:>5.1%} "
        f"{res['llm_judge_accuracy']:>7.1%}"
    )

with open("eval_results.json", "w") as f:
    json.dump(scored, f, indent=2)


  Evaluating: 4O-MINI → gpt-4o-mini
[01] EM: 1 | Judge: 1 — The predicted answer 'yes' correctly matches the gold answer and indicates that Scott Derrickson and Ed Wood were of the same nationality.
[02] EM: 0 | Judge: 0 — The predicted answer 'ambassador' does not match the gold answer 'Chief of Protocol' in meaning.
[03] EM: 1 | Judge: 1 — The predicted answer 'Animorphs' correctly matches the gold answer and answers the question accurately.
[04] EM: 1 | Judge: 1 — The predicted answer correctly matches the gold answer in meaning.
[05] EM: 0 | Judge: 1 — The predicted answer 'Greenwich Village' matches the gold answer in meaning, as it is a neighborhood in New York City.
[06] EM: 1 | Judge: 1 — The predicted answer 'YG Entertainment' correctly matches the gold answer and answers the question accurately.
[07] EM: 1 | Judge: 1 — The predicted answer 'Eenasul Fateh' matches the gold answer and correctly identifies the individual known by the stage name Aladin.
[08] EM: 0 | Judge: 1 — T

## Pre-Filtered KG-RAG

In [ ]:
def extract_triples(text: str) -> list[dict]:
    try:
        response = client.chat.completions.create(
            model=EXTRACTION_MODEL,
            max_completion_tokens=2500,
            temperature=0.0,
            response_format={"type": "json_object"},
            messages=[{
                "role": "user",
                "content": EXTRACTION_PROMPT.format(text=text)
            }]
        )
        raw = json.loads(response.choices[0].message.content)
        return raw.get("triples", []) # Safely extract the list
    
    except Exception as e:
        print(f"      [!] Extraction failed: {e}")
        return []

Knowledge Graph Construction

In [ ]:
def build_kg(paragraphs: list[dict]) -> nx.DiGraph:
    G = nx.DiGraph()
    for para in paragraphs:
        triples = extract_triples(para["text"])
        for t in triples:
            s = t.get("subject", "").strip()
            r = t.get("relation", "").strip()
            o = t.get("object",  "").strip()
            if s and r and o:
                G.add_edge(s, o, relation=r, source=para["text"])  # ← store source
    return G

Embeddings and Cosine Similarity

In [ ]:
def get_embeddings(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return np.array([d.embedding for d in response.data])

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b_norm = b / (np.linalg.norm(b) + 1e-9)
    return a_norm @ b_norm

Graph Retrieval

In [ ]:
def get_context_from_graph(G: nx.DiGraph, question: str, max_triples: int = 40) -> str:
    if len(G.edges) == 0:
        return ""

    nodes      = list(G.nodes)
    embeddings = get_embeddings(nodes + [question])
    node_embs  = embeddings[:-1]
    q_emb      = embeddings[-1]
    sims       = cosine_similarity(node_embs, q_emb)
    node_score = {node: float(sim) for node, sim in zip(nodes, sims)}

    # Node Scoring + Edge Ranking
    scored_edges = []
    for s, o, data in G.edges(data=True):
        score = node_score.get(s, 0) + node_score.get(o, 0)
        scored_edges.append((score, s, data.get("relation", "related to"), o, data.get("source", "")))

    scored_edges.sort(reverse=True)
    top_edges = scored_edges[:max_triples]

    # Context Construction
    seen, lines = set(), []
    for _, s, r, o, source in top_edges:
        if source and source not in seen:
            lines.append(source)
            seen.add(source)
        elif not source:
            lines.append(f"{s} {r} {o}")  # fallback if source missing

    return "\n".join(lines)

Answer Generation

In [ ]:
def llm_call_with_context(question: str, context: str, model_key: str) -> str:
    cfg   = MODELS[model_key]
    model = MODELS[model_key]["model"]
    reasoning_lvl = cfg["reasoning_lvl"]

    user_msg = f"""Context (knowledge graph triples):
{context}

Question: {question}"""

    if reasoning_lvl is not None:
        response = client.chat.completions.create(
            model=model,
            max_completion_tokens=2000,
            reasoning_effort=reasoning_lvl,
            messages=[
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user",   "content": user_msg},
            ]
        )
    else:
        response = client.chat.completions.create(
            model=model,
            max_completion_tokens=2000,
            temperature=0.0,
            messages=[
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user",   "content": user_msg},
            ]
        )
    return response.choices[0].message.content.strip()

Pre-Filtering

In [ ]:
def pre_filter_paragraphs(paragraphs: list[dict], question: str, top_k: int) -> list[dict]:
    if not paragraphs:
        return []

    texts = [p["text"] for p in paragraphs]

    embeddings = get_embeddings(texts + [question])
    para_embs = embeddings[:-1]
    q_emb     = embeddings[-1]

    sims = cosine_similarity(para_embs, q_emb)

    scored_paras = list(zip(sims, paragraphs))
    scored_paras.sort(key=lambda x: x[0], reverse=True)

    return [p for score, p in scored_paras[:top_k]]

Running the API call on all three models

In [53]:
def run_all_kg(dataset: list[dict]) -> dict:
    all_results = {}

    for model_key in MODELS:
        cfg = MODELS[model_key]
        print(f"\n{'='*60}")
        print(f"  {model_key.upper()} → {cfg['model']}  (reasoning: {cfg['reasoning_lvl'] or 'off'})")
        print(f"{'='*60}")

        preds = []

        for i, item in enumerate(dataset):
            q    = item["Q"]
            gold = item["A"]

            try:
                filtered_paragraphs = pre_filter_paragraphs(item["paragraphs"], q, top_k=2)

                G = build_kg(filtered_paragraphs)
                context = get_context_from_graph(G, q)

                if not context:
                    context = "No relevant triples extracted."

                pred = llm_call_with_context(q, context, model_key)

            except Exception as e:
                print(f"  [ERROR Q{i+1}] {e}")
                pred    = ""
                context = ""

            preds.append({
                "question"  : q,
                "gold"      : gold,
                "predicted" : pred,
                "context"   : context,   # saved for debugging
            })

            print(f"[{i+1:02d}] Q: {q[:65]}")
            print(f"      Gold: {gold}")
            print(f"      Pred: {pred[:80]}\n")

        all_results[model_key] = {
            "model"         : cfg["model"],
            "reasoning_lvl" : cfg["reasoning_lvl"] or "off",
            "predictions"   : preds,
        }

    return all_results

all_results = run_all_kg(my_ds)

with open("kg_predictions.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Saved → kg_predictions.json")


  4O-MINI → gpt-4o-mini  (reasoning: off)


[01] Q: Were Scott Derrickson and Ed Wood of the same nationality?
      Gold: yes
      Pred: yes

[02] Q: What government position was held by the woman who portrayed Corl
      Gold: Chief of Protocol
      Pred: no

[03] Q: What science fantasy young adult series, told in first person, ha
      Gold: Animorphs
      Pred: Animorphs

[04] Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same
      Gold: no
      Pred: no

[05] Q: The director of the romantic comedy "Big Stone Gap" is based in w
      Gold: Greenwich Village, New York City
      Pred: no

[06] Q: 2014 S/S is the debut album of a South Korean boy group that was 
      Gold: YG Entertainment
      Pred: YG Entertainment

[07] Q: Who was known by his stage name Aladin and helped organizations i
      Gold: Eenasul Fateh
      Pred: Eenasul Fateh

[08] Q: The arena where the Lewiston Maineiacs played their home games ca
      Gold: 3,677 seated
      Pred: 3677

[09] Q: Who is older, Annie Morton or Terry 

Evaluation

In [54]:
scored = evaluate("kg_predictions.json")

# Summary table
print(f"\n{'='*60}")
print(f"  BASELINE SUMMARY")
print(f"{'='*60}")
print(f"{'Tier':<12} {'Model':<16} {'Reasoning':<12} {'EM':>6} {'Judge':>7}")
print(f"{'-'*55}")
for key, res in scored.items():
    print(
        f"{key:<12} {res['model']:<16} {res['reasoning_lvl']:<12}"
        f"{res['exact_match_accuracy']:>5.1%} "
        f"{res['llm_judge_accuracy']:>7.1%}"
    )

with open("kg_eval_results.json", "w") as f:
    json.dump(scored, f, indent=2)

print("\nSaved")


  Evaluating: 4O-MINI → gpt-4o-mini
[01] EM: 1 | Judge: 1 — The predicted answer correctly matches the gold answer in meaning, as both Scott Derrickson and Ed Wood were American.
[02] EM: 0 | Judge: 0 — The predicted answer 'no' does not match the gold answer 'Chief of Protocol' in meaning.
[03] EM: 1 | Judge: 1 — The predicted answer 'Animorphs' correctly matches the gold answer and answers the question.
[04] EM: 1 | Judge: 1 — The predicted answer correctly matches the gold answer in meaning.
[05] EM: 0 | Judge: 0 — The predicted answer 'no' does not match the gold answer 'Greenwich Village, New York City' in meaning.
[06] EM: 1 | Judge: 1 — The predicted answer 'YG Entertainment' correctly matches the gold answer and answers the question accurately.
[07] EM: 1 | Judge: 1 — The predicted answer 'Eenasul Fateh' matches the gold answer and correctly identifies the individual known by the stage name Aladin.
[08] EM: 0 | Judge: 1 — The predicted answer '3677' matches the gold answer '3,